# MEG Faces Main Analysis

Using the **Wakeman–Henson multimodal face-processing dataset**, this tutorial asks how the MEG response evolves for **Famous faces**, **Unfamiliar faces**, and **Scrambled images**. We place all three conditions in one shared PCA space, then examine two planned comparisons without refitting the axes:

1. **Famous vs Unfamiliar** — does familiarity alter the trajectory?
2. **Faces vs Scrambled** — does face structure alter the trajectory?

<div class="alert alert-secondary">
<b>Position in the tutorial series.</b> The EEGBCI notebook introduces the core workflow. This MEG tutorial repeats that workflow with three additions: noise-whitened MEG sensors, equal-participant group summaries, and planned trajectory contrasts.
</div>

We use all six prepared participants. The analysis is descriptive and sensor-space based; it is not a source-localisation analysis.

## 0 — Setup

This notebook reads all six participants already prepared under the local MNE data directory. Downloading and preprocessing are deliberately kept outside this notebook.

<div class="alert alert-info">
<b>Analysis choices.</b> Set <code>SENSOR_SET</code> to <code>"all_sensors"</code>, <code>"sensors_occipital"</code>, <code>"sensors_temporal"</code>, or <code>"sensors_occipito_temporal"</code>. Set <code>METRIC_PCA_MODE</code> to <code>"shared"</code>, <code>"subject"</code>, or <code>"both"</code>. The regional choices are VectorView helmet selections, not source-localized cortical ROIs.
</div>

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from coco_pipe.dim_reduction import DimReduction
from coco_pipe.viz.interactive import plot_scree, plot_trajectory
from pca_neural_trajectories.wakeman_henson import (
    LABEL_NAMES,
    MEG_SENSOR_SETS,
    _load_wakeman_henson_container,
)

SEED = 42
DERIVATIVES_ROOT = (
    Path.home()
    / "mne_data"
    / "ds000117"
    / "derivatives"
    / "pca_trajectories"
)
SUBJECTS = ("01", "02", "03", "04", "05", "06")
SENSOR_SET = "all_sensors"

# Controls the coordinate system used for participant-level metrics.
# Choose "shared", "subject", or "both".
METRIC_PCA_MODE = "both"

N_COMPONENTS = 10
N_DISPLAY_COMPONENTS = 3
ACTIVE_WINDOW = (0.0, 0.6)

CONDITION_COLORS = {1: "#0072B2", 2: "#D55E00", 3: "#009E73"}
CONDITION_FILLS = {
    1: "rgba(0, 114, 178, 0.15)",
    2: "rgba(213, 94, 0, 0.15)",
    3: "rgba(0, 158, 115, 0.15)",
}
CONTRAST_COLORS = {
    "Faces vs Scrambled": "#7B2CBF",
    "Famous vs Unfamiliar": "#D55E00",
}
CONTRAST_FILLS = {
    "Faces vs Scrambled": "rgba(123, 44, 191, 0.15)",
    "Famous vs Unfamiliar": "rgba(213, 94, 0, 0.15)",
}

if METRIC_PCA_MODE not in {"shared", "subject", "both"}:
    raise ValueError('METRIC_PCA_MODE must be "shared", "subject", or "both".')
if SENSOR_SET not in MEG_SENSOR_SETS:
    raise ValueError(f"SENSOR_SET must be one of {tuple(MEG_SENSOR_SETS)}.")

pio.templates.default = "plotly_white"
rng = np.random.default_rng(SEED)

## 1 — Define the neural state

At each time point, the MEG state is the pattern across the selected sensors:

$$\mathbf{x}(t) \in \mathbb{R}^{N_{\mathrm{sensors}}}.$$

The preprocessing pipeline whitens each participant with that participant's empty-room covariance. Whitening places magnetometers and gradiometers on a common noise scale before PCA. We therefore **do not z-score the channels again**: doing so would discard the measured noise scaling.

<div class="alert alert-warning">
<b>Interpretation boundary.</b> Sensor-space PCs are mixtures of already mixed neural sources. We interpret trajectory geometry and timing, not a PC as a single anatomical generator.
</div>

## 2 — Load and inspect six participants

The loader returns a tensor with shape <code>trial × sensor × time</code>. Labels remain attached at the trial level, including repetition metadata for later auditing.

In [ ]:
available_subjects = tuple(
    path.name.removeprefix("sub-")
    for path in sorted(DERIVATIVES_ROOT.glob("sub-*"))
    if path.is_dir()
)
print(f"Prepared locally: {', '.join(available_subjects)}")
print(f"Selected now    : {', '.join(SUBJECTS)}")

container = _load_wakeman_henson_container(
    DERIVATIVES_ROOT,
    subjects=SUBJECTS,
    sensor_set=SENSOR_SET,
)

X = np.asarray(container.X, dtype=np.float32)
times = np.asarray(container.coords["time"], dtype=float)
channels = np.asarray(container.coords["channel"]).astype(str)
subjects = np.asarray(container.coords["subject"]).astype(str)
labels = np.asarray(container.y, dtype=int)
repetitions = np.asarray(container.coords["repetition"]).astype(str)

unique_subjects = np.unique(subjects)
conditions = np.array(sorted(LABEL_NAMES))

print(f"data shape : {X.shape}  (trials, sensors, time)")
print(f"time window: {times[0]:+.3f} to {times[-1]:+.3f} s")
print(f"sampling   : {1 / np.diff(times).mean():.1f} Hz")
print(f"subjects   : {', '.join(unique_subjects)}")
print(f"sensor set : {SENSOR_SET} ({len(channels)} sensors)")
print(f"whitening  : {container.meta.get('whitening', 'not recorded')}")

In [ ]:
condition_names = pd.Series([LABEL_NAMES[c] for c in labels], name="condition")
trial_counts = pd.crosstab(
    pd.Series(subjects, name="subject"),
    condition_names,
).reindex(columns=[LABEL_NAMES[c] for c in conditions])

print("Retained trials per participant and condition")
display(trial_counts)

print("Repetition counts")
display(pd.crosstab(condition_names, repetitions))

### Sensor-space sanity check

Before reducing dimensionality, verify that stimulus-locked structure is visible in the whitened sensor data. For each participant and condition, we average trials and compute the root-mean-square response across sensors. This is a compact quality-control view, not a replacement for inspecting the original epochs.

In [ ]:
sensor_magnitude = {c: [] for c in conditions}

for subject in unique_subjects:
    for condition in conditions:
        rows = (subjects == subject) & (labels == condition)
        evoked = X[rows].mean(axis=0)
        sensor_magnitude[condition].append(np.sqrt(np.mean(evoked**2, axis=0)))

fig_sensor = go.Figure()
for condition in conditions:
    curves = np.asarray(sensor_magnitude[condition])
    mean = curves.mean(axis=0)
    sem = curves.std(axis=0, ddof=1) / np.sqrt(len(curves))
    color = CONDITION_COLORS[condition]
    name = LABEL_NAMES[condition]
    fig_sensor.add_trace(
        go.Scatter(x=times, y=mean, mode="lines", name=name, line=dict(color=color, width=3))
    )
    fig_sensor.add_trace(
        go.Scatter(
            x=times, y=mean + sem, mode="lines", line=dict(width=0),
            showlegend=False, hoverinfo="skip",
        )
    )
    fig_sensor.add_trace(
        go.Scatter(
            x=times, y=mean - sem, mode="lines", line=dict(width=0),
            fill="tonexty", fillcolor=CONDITION_FILLS[condition],
            showlegend=False, hoverinfo="skip",
        )
    )

fig_sensor.add_vline(x=0, line_dash="dash", line_color="black")
fig_sensor.update_layout(
    title="Whitened sensor-space response magnitude",
    xaxis_title="Time (s)",
    yaxis_title="RMS response (noise-normalised units)",
    legend_title_text="Condition",
)
fig_sensor.show()

## 3 — Reshape trials into PCA observations

PCA expects <code>observations × features</code>. Here, one observation is one time point from one trial, and the features are MEG sensors:

$$(\text{trials},\ \text{sensors},\ \text{time}) \rightarrow
(\text{trials} \times \text{time},\ \text{sensors}).$$

The fit ignores condition labels. It captures variance shared across the full three-condition dataset. Because all retained observations enter the fit, participants with more retained trials contribute slightly more to the PCA basis; the count table above makes that weighting visible. Group summaries later give each participant equal weight.

In [ ]:
n_trials, n_sensors, n_times = X.shape
pooled = X.transpose(0, 2, 1).reshape(n_trials * n_times, n_sensors)

print(f"input tensor : {X.shape}")
print(f"PCA matrix   : {pooled.shape}  (trial-time observations, sensors)")

## 4 — Fit one shared PCA

A single PCA basis is fitted across all participants and conditions. Every trajectory and contrast below therefore uses the same axes. We fit ten components so the dimensionality can be inspected, but use the first three for the main figures.

<div class="alert alert-info">
<b>Three PCs are a display choice.</b> The scree plot and cumulative variance show how much structure the 3-D view omits. A clean 3-D figure is not evidence that the full MEG response is intrinsically three-dimensional.
</div>

In [ ]:
shared_pca = DimReduction(
    method="PCA",
    n_components=N_COMPONENTS,
    random_state=SEED,
)
scores_flat = shared_pca.fit_transform(pooled)

diagnostics = shared_pca.get_diagnostics()
explained_variance = np.asarray(diagnostics["explained_variance_ratio_"])

print(f"Variance explained by 3 PCs : {explained_variance[:3].sum():.1%}")
print(f"Variance explained by 10 PCs: {explained_variance.sum():.1%}")
if diagnostics.get("participation_ratio_") is not None:
    print(f"Participation ratio (10-PC spectrum): {diagnostics['participation_ratio_']:.2f}")

fig_scree = plot_scree(explained_variance)
fig_scree.update_layout(title="Shared PCA: explained and cumulative variance")
fig_scree.show()

## 5 — Reconstruct and baseline the trajectories

The PCA scores are reshaped back to <code>trial × time × component</code>. We then subtract each trial's mean pre-stimulus score from every time point. This anchors the trajectories to their own baseline in PC space.

A shared PCA has one sign convention for the entire dataset, so no participant-specific sign alignment is needed.

In [ ]:
scores = scores_flat.reshape(n_trials, n_times, N_COMPONENTS)
baseline_mask = times < 0
scores_baselined = scores - scores[:, baseline_mask].mean(axis=1, keepdims=True)

print(f"flat scores       : {scores_flat.shape}")
print(f"trial trajectories: {scores_baselined.shape}")
print(
    "Mean absolute baseline offset after correction: "
    f"{np.abs(scores_baselined[:, baseline_mask].mean(axis=1)).mean():.3e}"
)

### 5a — Fit one PCA per participant

This reproduces the subject-level strategy used in the EEGBCI tutorial. Each participant receives a separate PCA fitted to all three conditions. We never average the resulting PC coordinates across participants because PC1 for one participant is not the same axis as PC1 for another.

Euclidean separation and speed are invariant to sign flips and rotations within a retained subspace, so sign alignment is unnecessary for these metrics. The important sensitivity is **subspace selection**: each participant's top three PCs may retain different variance from the shared top three PCs.

In [ ]:
subject_scores_baselined = None
subject_pcas = {}

if METRIC_PCA_MODE in {"subject", "both"}:
    subject_scores_baselined = np.empty_like(scores_baselined)

    for subject in unique_subjects:
        rows = subjects == subject
        X_subject = X[rows]
        n_subject_trials = X_subject.shape[0]
        subject_matrix = X_subject.transpose(0, 2, 1).reshape(
            n_subject_trials * n_times, n_sensors
        )

        subject_pca = DimReduction(
            method="PCA",
            n_components=N_COMPONENTS,
            random_state=SEED,
        )
        subject_flat = subject_pca.fit_transform(subject_matrix)
        subject_traj = subject_flat.reshape(
            n_subject_trials, n_times, N_COMPONENTS
        )
        subject_traj -= subject_traj[:, baseline_mask].mean(
            axis=1, keepdims=True
        )

        subject_pcas[subject] = subject_pca
        subject_scores_baselined[rows] = subject_traj

    subject_evr = pd.DataFrame({
        subject: subject_pcas[subject]
        .get_diagnostics()["explained_variance_ratio_"][:3]
        for subject in unique_subjects
    }, index=["PC1", "PC2", "PC3"]).T
    subject_evr["cumulative_3pc"] = subject_evr.sum(axis=1)
    display(subject_evr.round(3))
else:
    print("Subject-level PCA skipped: METRIC_PCA_MODE is set to shared.")

### Read the PCs before reading the geometry

PC traces reveal when each axis changes. The loading table shows which sensors contribute most strongly, but it does not provide source localisation. Component signs are arbitrary; timing and relative geometry are the stable quantities.

In [ ]:
subject_condition = {}
for subject in unique_subjects:
    for condition in conditions:
        rows = (subjects == subject) & (labels == condition)
        subject_condition[subject, condition] = scores_baselined[rows].mean(axis=0)

fig_pc = make_subplots(
    rows=N_DISPLAY_COMPONENTS,
    cols=1,
    shared_xaxes=True,
    subplot_titles=[f"PC{i + 1}" for i in range(N_DISPLAY_COMPONENTS)],
    vertical_spacing=0.08,
)

for pc in range(N_DISPLAY_COMPONENTS):
    for condition in conditions:
        curves = np.stack([subject_condition[s, condition][:, pc] for s in unique_subjects])
        mean = curves.mean(axis=0)
        sem = curves.std(axis=0, ddof=1) / np.sqrt(len(curves))
        color = CONDITION_COLORS[condition]
        name = LABEL_NAMES[condition]
        fig_pc.add_trace(
            go.Scatter(
                x=times, y=mean, mode="lines", name=name, legendgroup=name,
                showlegend=pc == 0, line=dict(color=color, width=2.5),
            ),
            row=pc + 1, col=1,
        )
        fig_pc.add_trace(
            go.Scatter(
                x=times, y=mean + sem, mode="lines", line=dict(width=0),
                legendgroup=name, showlegend=False, hoverinfo="skip",
            ),
            row=pc + 1, col=1,
        )
        fig_pc.add_trace(
            go.Scatter(
                x=times, y=mean - sem, mode="lines", line=dict(width=0),
                fill="tonexty", fillcolor=CONDITION_FILLS[condition], legendgroup=name,
                showlegend=False, hoverinfo="skip",
            ),
            row=pc + 1, col=1,
        )

fig_pc.add_vline(x=0, line_dash="dash", line_color="black")
fig_pc.update_xaxes(title_text="Time (s)", row=N_DISPLAY_COMPONENTS, col=1)
fig_pc.update_yaxes(title_text="Score (a.u.)")
fig_pc.update_layout(height=720, title="Principal-component time courses")
fig_pc.show()

In [ ]:
loadings = np.asarray(shared_pca.get_components())[:N_DISPLAY_COMPONENTS]
loading_rows = []

for pc, weights in enumerate(loadings, start=1):
    strongest = np.argsort(np.abs(weights))[-8:][::-1]
    loading_rows.extend(
        {"component": f"PC{pc}", "sensor": channels[i], "loading": weights[i]}
        for i in strongest
    )

print("Sensors with the largest absolute loading on each displayed PC")
display(pd.DataFrame(loading_rows).round({"loading": 3}))

## 6 — Plot all three conditions in the shared space

Trials are first averaged within each participant and condition. We then average those participant trajectories, so each participant contributes equally even if trial counts differ. The shaded 2-D envelope is the SEM across the three participant means.

<div class="alert alert-info">
<b>How to read the geometry.</b> Follow the trajectories from the pre-stimulus cluster through the post-stimulus excursion, then ask when the conditions diverge. A loop or bend is a geometric description; it does not by itself establish an oscillatory neural mechanism.
</div>

In [ ]:
group_trajectories = np.stack([
    np.stack([subject_condition[s, c] for s in unique_subjects]).mean(axis=0)
    for c in conditions
])
group_sem = np.stack([
    np.stack([subject_condition[s, c] for s in unique_subjects]).std(axis=0, ddof=1)
    / np.sqrt(len(unique_subjects))
    for c in conditions
])

plot_labels = np.array([LABEL_NAMES[c] for c in conditions])
color_map = {LABEL_NAMES[c]: CONDITION_COLORS[c] for c in conditions}

fig_2d = plot_trajectory(
    X=group_trajectories[..., :2],
    times=times,
    labels=plot_labels,
    color_map=color_map,
    title="Equal-participant mean trajectories: PC1–PC2",
    dimensions=2,
    smooth_window=12,
    show_markers=False,
    add_start_end_markers=True,
)
fig_2d.show()

In [ ]:
fig_3d = plot_trajectory(
    X=group_trajectories[..., :3],
    times=times,
    labels=plot_labels,
    color_map=color_map,
    smooth_window=12,
    show_markers=False,
    linewidth=10,                      # Auto-scales all trace line widths
    title="Equal-participant mean trajectories: PC1–PC2–PC3",
    dimensions=3,
    add_start_end_markers=True,
    height=700,
)
fig_3d.show()

## 7 — Evaluate the planned contrasts

Both contrasts are evaluated per participant:

- **Famous vs Unfamiliar** is the Euclidean distance between those condition means.
- **Faces vs Scrambled** first averages Famous and Unfamiliar with equal weight, then measures distance to Scrambled.

The switch determines whether these participant-level distances are computed in the common shared basis, each participant's own PCA basis, or both. We subtract each participant's mean pre-stimulus distance. A convincing effect should emerge after stimulus onset and appear in more than one participant—not only in the thick group-average line.

In [ ]:
def compute_contrast_curves(trajectories, trial_labels):
    # Return baseline-relative separation curves for each participant.
    curves = {"Faces vs Scrambled": [], "Famous vs Unfamiliar": []}
    for subject in unique_subjects:
        means = {
            condition: trajectories[
                (subjects == subject) & (trial_labels == condition)
            ].mean(axis=0)
            for condition in conditions
        }
        faces = 0.5 * (means[1] + means[2])
        distances = {
            "Faces vs Scrambled": np.linalg.norm(faces - means[3], axis=1),
            "Famous vs Unfamiliar": np.linalg.norm(
                means[1] - means[2], axis=1
            ),
        }
        for name, distance in distances.items():
            curves[name].append(distance - distance[baseline_mask].mean())
    return {name: np.asarray(values) for name, values in curves.items()}


metric_trajectory_spaces = {}
if METRIC_PCA_MODE in {"shared", "both"}:
    metric_trajectory_spaces["Shared PCA"] = scores_baselined[
        ..., :N_DISPLAY_COMPONENTS
    ]
if METRIC_PCA_MODE in {"subject", "both"}:
    metric_trajectory_spaces["Subject PCA"] = subject_scores_baselined[
        ..., :N_DISPLAY_COMPONENTS
    ]

contrast_curves_by_space = {
    space: compute_contrast_curves(trajectories, labels)
    for space, trajectories in metric_trajectory_spaces.items()
}

In [ ]:
n_spaces = len(contrast_curves_by_space)
contrast_names = ["Faces vs Scrambled", "Famous vs Unfamiliar"]
fig_contrasts = make_subplots(
    rows=n_spaces,
    cols=2,
    shared_xaxes=True,
    shared_yaxes=True,
    subplot_titles=[
        f"{space}: {contrast}"
        for space in contrast_curves_by_space
        for contrast in contrast_names
    ],
)

for row, (space, space_curves) in enumerate(
    contrast_curves_by_space.items(), start=1
):
    for column, name in enumerate(contrast_names, start=1):
        curves = space_curves[name]
        color = CONTRAST_COLORS[name]
        for subject, curve in zip(unique_subjects, curves):
            fig_contrasts.add_trace(
                go.Scatter(
                    x=times,
                    y=curve,
                    mode="lines",
                    line=dict(color=color, width=1),
                    opacity=0.28,
                    name=f"sub-{subject}",
                    showlegend=False,
                ),
                row=row,
                col=column,
            )
        mean = curves.mean(axis=0)
        sem = curves.std(axis=0, ddof=1) / np.sqrt(len(curves))
        fig_contrasts.add_trace(
            go.Scatter(
                x=times,
                y=mean,
                mode="lines",
                line=dict(color=color, width=4),
                name=f"{space}: {name}",
                showlegend=False,
            ),
            row=row,
            col=column,
        )
        fig_contrasts.add_trace(
            go.Scatter(
                x=times,
                y=mean + sem,
                mode="lines",
                line=dict(width=0),
                showlegend=False,
                hoverinfo="skip",
            ),
            row=row,
            col=column,
        )
        fig_contrasts.add_trace(
            go.Scatter(
                x=times,
                y=mean - sem,
                mode="lines",
                line=dict(width=0),
                fill="tonexty",
                fillcolor=CONTRAST_FILLS[name],
                showlegend=False,
                hoverinfo="skip",
            ),
            row=row,
            col=column,
        )

fig_contrasts.add_vline(x=0, line_dash="dash", line_color="black")
fig_contrasts.add_hline(y=0, line_dash="dot", line_color="grey")
fig_contrasts.update_xaxes(title_text="Time (s)")
fig_contrasts.update_yaxes(
    title_text="Distance relative to baseline (a.u.)", col=1
)
fig_contrasts.update_layout(
    height=390 * n_spaces,
    title="Planned contrasts across PCA metric spaces",
)
fig_contrasts.show()

In [ ]:
active_mask = (times >= ACTIVE_WINDOW[0]) & (times <= ACTIVE_WINDOW[1])
active_times = times[active_mask]
summary_rows = []

for space, space_curves in contrast_curves_by_space.items():
    for name, curves in space_curves.items():
        for subject, curve in zip(unique_subjects, curves):
            active_curve = curve[active_mask]
            peak_index = int(np.argmax(active_curve))
            summary_rows.append({
                "pca_space": space,
                "subject": subject,
                "contrast": name,
                "auc_0_600ms": np.trapezoid(active_curve, active_times),
                "peak_separation": active_curve[peak_index],
                "peak_time_s": active_times[peak_index],
            })

contrast_summary = pd.DataFrame(summary_rows)
display(contrast_summary.round(3))
display(
    contrast_summary.groupby(["pca_space", "contrast"])
    .agg(
        mean_auc=("auc_0_600ms", "mean"),
        sem_auc=("auc_0_600ms", "sem"),
        mean_peak_time_s=("peak_time_s", "mean"),
    )
    .round(3)
)

## 8 — Compare the planned effects with a within-participant null

PCA was fitted without labels, so its axes remain fixed. We now shuffle condition labels within each participant, preserving that participant's trial count and preprocessing history. Each shuffle recomputes both planned contrasts.

Two hundred permutations keep this three-participant tutorial practical and limit the smallest attainable empirical p-value to 1 / 201 ≈ 0.005. Read the histogram by locating the observed AUC relative to the shuffled distribution. A final analysis should use more permutations and more participants.

In [ ]:
N_PERMUTATIONS = 200
observed_auc = {}
null_auc = {}

for space, trajectories in metric_trajectory_spaces.items():
    observed_auc[space] = {
        name: np.trapezoid(curves.mean(axis=0)[active_mask], active_times)
        for name, curves in contrast_curves_by_space[space].items()
    }
    null_auc[space] = {
        name: [] for name in contrast_curves_by_space[space]
    }

for _ in range(N_PERMUTATIONS):
    shuffled = labels.copy()
    for subject in unique_subjects:
        rows = np.flatnonzero(subjects == subject)
        shuffled[rows] = rng.permutation(shuffled[rows])

    for space, trajectories in metric_trajectory_spaces.items():
        permuted_curves = compute_contrast_curves(trajectories, shuffled)
        for name, curves in permuted_curves.items():
            null_auc[space][name].append(
                np.trapezoid(
                    curves.mean(axis=0)[active_mask], active_times
                )
            )

inference_rows = []
for space, space_nulls in null_auc.items():
    for name, values in space_nulls.items():
        values = np.asarray(values)
        null_auc[space][name] = values
        inference_rows.append({
            "pca_space": space,
            "contrast": name,
            "observed_auc": observed_auc[space][name],
            "null_mean": values.mean(),
            "empirical_p": (
                1 + np.sum(values >= observed_auc[space][name])
            ) / (N_PERMUTATIONS + 1),
        })

inference = pd.DataFrame(inference_rows)
display(inference.round(4))

In [ ]:
fig_null = make_subplots(
    rows=len(null_auc),
    cols=2,
    subplot_titles=[
        f"{space}: {contrast}"
        for space in null_auc
        for contrast in contrast_names
    ],
)

for row, (space, space_nulls) in enumerate(null_auc.items(), start=1):
    for column, name in enumerate(contrast_names, start=1):
        values = space_nulls[name]
        color = CONTRAST_COLORS[name]
        fig_null.add_trace(
            go.Histogram(
                x=values,
                nbinsx=25,
                marker_color=color,
                opacity=0.72,
                showlegend=False,
            ),
            row=row,
            col=column,
        )
        fig_null.add_vline(
            x=observed_auc[space][name],
            line_color="black",
            line_width=3,
            annotation_text="observed",
            row=row,
            col=column,
        )

fig_null.update_xaxes(title_text="Baseline-relative separation AUC")
fig_null.update_yaxes(title_text="Permutations", col=1)
fig_null.update_layout(
    height=360 * len(null_auc),
    title="Within-participant permutation nulls",
)
fig_null.show()

## 9 — Describe trajectory speed

Speed is the rate of movement through PCA space. It can reveal a shared state transition, but it is not automatically condition-specific. In many neural datasets the largest dynamical component is common across conditions, so similar speed curves are scientifically informative rather than a failed result.

We compute speed from each participant's condition-mean trajectory, then summarize across participants. Numerical derivatives amplify noise; interpret broad changes rather than isolated samples.

In [ ]:
fig_speed = make_subplots(
    rows=len(metric_trajectory_spaces),
    cols=1,
    shared_xaxes=True,
    subplot_titles=list(metric_trajectory_spaces),
)

for row, (space, trajectories) in enumerate(
    metric_trajectory_spaces.items(), start=1
):
    for condition in conditions:
        speed_curves = []
        for subject in unique_subjects:
            subject_mean = trajectories[
                (subjects == subject) & (labels == condition)
            ].mean(axis=0)
            velocity = np.gradient(subject_mean, times, axis=0)
            speed_curves.append(np.linalg.norm(velocity, axis=1))

        speed_curves = np.asarray(speed_curves)
        mean = speed_curves.mean(axis=0)
        sem = speed_curves.std(axis=0, ddof=1) / np.sqrt(len(speed_curves))
        color = CONDITION_COLORS[condition]
        name = LABEL_NAMES[condition]
        fig_speed.add_trace(
            go.Scatter(
                x=times,
                y=mean,
                mode="lines",
                name=name,
                legendgroup=name,
                showlegend=row == 1,
                line=dict(color=color, width=3),
            ),
            row=row,
            col=1,
        )
        fig_speed.add_trace(
            go.Scatter(
                x=times,
                y=mean + sem,
                mode="lines",
                line=dict(width=0),
                showlegend=False,
                hoverinfo="skip",
            ),
            row=row,
            col=1,
        )
        fig_speed.add_trace(
            go.Scatter(
                x=times,
                y=mean - sem,
                mode="lines",
                line=dict(width=0),
                fill="tonexty",
                fillcolor=CONDITION_FILLS[condition],
                showlegend=False,
                hoverinfo="skip",
            ),
            row=row,
            col=1,
        )

fig_speed.add_vline(x=0, line_dash="dash", line_color="black")
fig_speed.update_xaxes(title_text="Time (s)")
fig_speed.update_yaxes(title_text="Speed (a.u./s)")
fig_speed.update_layout(
    height=360 * len(metric_trajectory_spaces),
    title="Trajectory speed across PCA metric spaces",
)
fig_speed.show()

## 10 — Check whether the shared three-PC result is fragile

Distances generally increase when more orthogonal dimensions are added, so raw AUC values from different component counts are not directly comparable. In the shared PCA, we instead ask whether the **timecourse shape** and peak timing are stable when the contrast is computed with 2, 3, or 5 PCs.

In [ ]:
reference_curves = {
    name: curves.mean(axis=0)
    for name, curves in compute_contrast_curves(
        scores_baselined[..., :3], labels
    ).items()
}
sensitivity_rows = []

for n_components in (2, 3, 5):
    candidate = compute_contrast_curves(
        scores_baselined[..., :n_components], labels
    )
    for name, curves in candidate.items():
        mean_curve = curves.mean(axis=0)
        active_curve = mean_curve[active_mask]
        sensitivity_rows.append({
            "contrast": name,
            "components": n_components,
            "r_with_3pc_timecourse": np.corrcoef(
                active_curve, reference_curves[name][active_mask]
            )[0, 1],
            "peak_time_s": active_times[int(np.argmax(active_curve))],
        })

sensitivity = pd.DataFrame(sensitivity_rows)
display(sensitivity.round(3))

## 11 — Fit focused PCA spaces for two-condition questions

The three-condition PCA is the primary analysis because every condition shares one coordinate system. We now fit two additional shared PCAs:

1. **Famous and Unfamiliar only**
2. **Famous and Scrambled only**

Each focused PCA is fitted across all selected participants but only the trials in that pair. This can reveal pair-relevant variance that receives less priority in the three-condition fit.

<div class="alert alert-warning">
<b>Do not compare axes across these models.</b> PC1 in one focused PCA is not PC1 in another. Compare the timing and consistency of divergence, not raw coordinates or raw distances between models.
</div>

In [ ]:
FOCUSED_PAIRS = {
    "Famous vs Unfamiliar": (1, 2),
    "Famous vs Scrambled": (1, 3),
}
focused_results = {}

for pair_name, pair in FOCUSED_PAIRS.items():
    pair_mask = np.isin(labels, pair)
    X_pair = X[pair_mask]
    labels_pair = labels[pair_mask]
    subjects_pair = subjects[pair_mask]
    n_pair_trials = X_pair.shape[0]
    pair_matrix = X_pair.transpose(0, 2, 1).reshape(
        n_pair_trials * n_times, n_sensors
    )

    pair_pca = DimReduction(
        method="PCA",
        n_components=N_COMPONENTS,
        random_state=SEED,
    )
    pair_flat = pair_pca.fit_transform(pair_matrix)
    pair_scores = pair_flat.reshape(
        n_pair_trials, n_times, N_COMPONENTS
    )
    pair_scores -= pair_scores[:, baseline_mask].mean(
        axis=1, keepdims=True
    )

    subject_pair_means = {
        (subject, condition): pair_scores[
            (subjects_pair == subject) & (labels_pair == condition)
        ].mean(axis=0)
        for subject in unique_subjects
        for condition in pair
    }
    pair_group = np.stack([
        np.stack([
            subject_pair_means[subject, condition]
            for subject in unique_subjects
        ]).mean(axis=0)
        for condition in pair
    ])
    pair_sem = np.stack([
        np.stack([
            subject_pair_means[subject, condition]
            for subject in unique_subjects
        ]).std(axis=0, ddof=1) / np.sqrt(len(unique_subjects))
        for condition in pair
    ])
    separation = np.stack([
        np.linalg.norm(
            subject_pair_means[subject, pair[0]][:, :3]
            - subject_pair_means[subject, pair[1]][:, :3],
            axis=1,
        )
        for subject in unique_subjects
    ])
    separation -= separation[:, baseline_mask].mean(
        axis=1, keepdims=True
    )

    focused_results[pair_name] = {
        "pair": pair,
        "pca": pair_pca,
        "scores": pair_scores,
        "group": pair_group,
        "sem": pair_sem,
        "separation": separation,
        "variance_3pc": pair_pca
        .get_diagnostics()["explained_variance_ratio_"][:3].sum(),
    }

focused_variance = pd.DataFrame([
    {
        "focused_space": pair_name,
        "variance_explained_3pc": result["variance_3pc"],
    }
    for pair_name, result in focused_results.items()
])
display(focused_variance.round(3))

focused_summary_rows = []
for pair_name, result in focused_results.items():
    for subject, curve in zip(unique_subjects, result["separation"]):
        active_curve = curve[active_mask]
        peak_index = int(np.argmax(active_curve))
        focused_summary_rows.append({
            "focused_space": pair_name,
            "subject": subject,
            "auc_0_600ms": np.trapezoid(active_curve, active_times),
            "peak_separation": active_curve[peak_index],
            "peak_time_s": active_times[peak_index],
        })

focused_summary = pd.DataFrame(focused_summary_rows)
display(focused_summary.round(3))
display(
    focused_summary.groupby("focused_space")
    .agg(
        mean_auc=("auc_0_600ms", "mean"),
        sem_auc=("auc_0_600ms", "sem"),
        mean_peak_time_s=("peak_time_s", "mean"),
    )
    .round(3)
)

In [ ]:
for pair_name, result in focused_results.items():
    pair = result["pair"]
    pair_labels = np.array([LABEL_NAMES[c] for c in pair])
    pair_colors = {LABEL_NAMES[c]: CONDITION_COLORS[c] for c in pair}

    figure = plot_trajectory(
        X=result["group"][..., :3],
        times=times,
        labels=pair_labels,
        color_map=pair_colors,
        title=f"Focused shared PCA: {pair_name}",
        dimensions=3,
        show_markers=False,
        smooth_window=12,
        add_start_end_markers=True,
    )
    figure.show()

In [ ]:
fig_focused_sep = make_subplots(
    rows=1,
    cols=2,
    shared_yaxes=True,
    subplot_titles=list(focused_results),
)

for column, (pair_name, result) in enumerate(
    focused_results.items(), start=1
):
    curves = result["separation"]
    color = CONTRAST_COLORS.get(pair_name, "#444444")
    for subject, curve in zip(unique_subjects, curves):
        fig_focused_sep.add_trace(
            go.Scatter(
                x=times,
                y=curve,
                mode="lines",
                line=dict(color=color, width=1),
                opacity=0.28,
                showlegend=False,
                name=f"sub-{subject}",
            ),
            row=1,
            col=column,
        )
    mean = curves.mean(axis=0)
    sem = curves.std(axis=0, ddof=1) / np.sqrt(len(curves))
    fig_focused_sep.add_trace(
        go.Scatter(
            x=times,
            y=mean,
            mode="lines",
            line=dict(color=color, width=4),
            showlegend=False,
            name=pair_name,
        ),
        row=1,
        col=column,
    )
    fig_focused_sep.add_trace(
        go.Scatter(
            x=times,
            y=mean + sem,
            mode="lines",
            line=dict(width=0),
            showlegend=False,
            hoverinfo="skip",
        ),
        row=1,
        col=column,
    )
    fig_focused_sep.add_trace(
        go.Scatter(
            x=times,
            y=mean - sem,
            mode="lines",
            line=dict(width=0),
            fill="tonexty",
            fillcolor=CONTRAST_FILLS.get(
                pair_name, "rgba(68, 68, 68, 0.15)"
            ),
            showlegend=False,
            hoverinfo="skip",
        ),
        row=1,
        col=column,
    )

fig_focused_sep.add_vline(
    x=0, line_dash="dash", line_color="black"
)
fig_focused_sep.add_hline(
    y=0, line_dash="dot", line_color="grey"
)
fig_focused_sep.update_xaxes(title_text="Time (s)")
fig_focused_sep.update_yaxes(
    title_text="Distance relative to baseline (a.u.)",
    row=1,
    col=1,
)
fig_focused_sep.update_layout(
    height=470,
    title="Focused shared-PCA separation",
)
fig_focused_sep.show()

### Interpret the focused spaces

Use these figures to ask whether a pair becomes easier to see when PCA is allowed to focus on that pair. Agreement in divergence timing with the three-condition space strengthens the result. A dramatic change restricted to a focused PCA suggests that the effect is weak relative to variance contributed by the third condition.

## 12 — Optional compact export

The tutorial is complete without writing artifacts. Enable this cell only when you want a lightweight hand-off containing the shared scores, coordinates, PCA loadings, and the participant-level contrast table.

In [ ]:
SAVE_RESULTS = False

if SAVE_RESULTS:
    output_dir = Path("outputs/tutorial_megfaces_main") / SENSOR_SET
    output_dir.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        output_dir / "shared_pca_trajectories.npz",
        scores=scores_baselined.astype(np.float32),
        times=times,
        labels=labels,
        subjects=subjects,
        channels=channels,
        sensor_set=SENSOR_SET,
        loadings=np.asarray(shared_pca.get_components()),
        explained_variance_ratio=explained_variance,
        subject_scores=(
            subject_scores_baselined.astype(np.float32)
            if subject_scores_baselined is not None else np.array([])
        ),
    )
    contrast_summary.to_csv(
        output_dir / "planned_contrasts.csv", index=False
    )
    inference.to_csv(output_dir / "within_subject_null.csv", index=False)
    print(f"Saved results to {output_dir}")
else:
    print("SAVE_RESULTS is False; nothing was written.")

## Takeaway

The notebook now answers the same questions in complementary coordinate systems:

- **One three-condition shared PCA** provides the common group trajectory.
- **Participant-specific PCAs** test whether separation and speed depend on using a shared basis.
- **Two focused shared PCAs** ask whether Famous–Unfamiliar or Famous–Scrambled structure is clearer when the third condition is excluded.

The PCA space and the statistical unit are separate choices. Group figures require shared axes, whereas metrics are always computed per participant before group summarisation.

<div class="alert alert-success">
<b>Next representation: band-limited envelopes.</b> Broadband ERF trajectories emphasize phase-locked responses. The spectral companion tutorial uses MNE filtering followed by the Hilbert amplitude envelope, then applies the same PCA logic to alpha, beta, or gamma activity.
</div>

With six participants, numerical results remain demonstrations—not population estimates. Sensor-space geometry also does not establish anatomical generators or mechanistic dynamics.